# T1 — Pipeline reprodutível: `dvc.yaml` e `dvc repro`

**Objetivo:** transformar "rodei uns scripts na ordem certa" em um pipeline declarado,
que só re-executa o que mudou e deixa um recibo (`dvc.lock`) de tudo que foi usado.

```
data/raw/bike.csv
   │
prepare  ─► data/prepared/bike.csv           ✔ já vem pronto
   │
split    ─► data/split/{train,test}.csv      ✔ já vem pronto       (params: split)
   │
train    ─► models/                          👥 Parte 1, juntos     (params: train, mlflow)
   │
evaluate ─► metrics.json                     🧑‍💻 Parte 2, sozinhos
plot     ─► reports/previsto_vs_real.png     🧑‍💻 Parte 2, sozinhos
```

| Chave do estágio | O que declara | Exemplo |
|---|---|---|
| `cmd` | o comando que o estágio roda | `python src/split.py` |
| `deps` | arquivos que ele **lê** (mudou um → roda de novo) | `src/split.py`, `data/prepared` |
| `params` | seções do `params.yaml` que ele usa | `split` |
| `outs` | o que ele **gera** (vai para o cache do DVC) | `data/split` |
| `metrics` | arquivo de métricas (`dvc metrics show/diff`) | `metrics.json` |
| `plots` | gráficos (`dvc plots show`) | `reports/previsto_vs_real.png` |

> 💡 As células `%%bash` rodam comandos de terminal. Se preferir, copie-os para o terminal.
> Os comandos que mudam o Git e o DVC vêm **comentados** (com `#`), para não rodarem sem querer.
> Quando chegar a hora, tire o `#` (selecione as linhas e use **Ctrl + /**) e rode a célula.

In [ ]:
import os, sys
from pathlib import Path

# Trabalha a partir da raiz do repositório (a pasta que contém o params.yaml)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "params.yaml").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

---
# Parte 1 · Juntos

## 1. Inicializar o DVC e configurar o remote

O `dvc init` cria a pasta `.dvc/` (configuração e cache). O **remote** é onde o DVC guarda os arquivos que ele
controla: nesta aula, uma **pasta local** (`../dvc-storage`) que simula um storage na nuvem.
Na Aula 2 trocamos por um remote de verdade (DagsHub).

In [ ]:
%%bash
# dvc init
# dvc remote add -d local ../dvc-storage
cat .dvc/config
# git add .dvc .dvcignore
# git commit -m "Inicializa DVC com remote local"

## 2. Ponto de partida: o pipeline com 2 estágios prontos

Os estágios leem os parâmetros do `params.yaml`:

In [ ]:
%%bash
cat params.yaml

A célula abaixo grava o `dvc.yaml` na raiz do projeto (abra o arquivo no editor para ver).
Leia os dois estágios com calma: **os próximos seguem o mesmo molde.**

In [ ]:
%%writefile dvc.yaml
# Pipeline do projeto. Cada estágio declara o comando (cmd), o que ele lê (deps),
# os parâmetros do params.yaml que usa (params) e o que ele gera (outs).
stages:
  prepare:
    cmd: python src/prepare.py
    deps:
    - src/prepare.py
    - src/bike.py
    - data/raw/bike.csv
    outs:
    - data/prepared

  split:
    cmd: python src/split.py
    deps:
    - src/split.py
    - src/bike.py
    - data/prepared
    params:
    - split
    outs:
    - data/split

In [ ]:
%%bash
dvc dag
# dvc repro

## 3. Escrever o estágio `train`

Completamos juntos o estágio de treino. A célula regrava o `dvc.yaml` inteiro (os 2 estágios prontos + `train`),
então pode rodá-la de novo sem medo de duplicar nada.

In [ ]:
%%writefile dvc.yaml
# Pipeline do projeto. Cada estágio declara o comando (cmd), o que ele lê (deps),
# os parâmetros do params.yaml que usa (params) e o que ele gera (outs).
stages:
  prepare:
    cmd: python src/prepare.py
    deps:
    - src/prepare.py
    - src/bike.py
    - data/raw/bike.csv
    outs:
    - data/prepared

  split:
    cmd: python src/split.py
    deps:
    - src/split.py
    - src/bike.py
    - data/prepared
    params:
    - split
    outs:
    - data/split

  train:
    # TODO: o comando que roda o treino
    cmd: ...
    deps:
    # TODO: o script, o bike.py e o arquivo de TREINO que o split gera
    - ...
    params:
    # TODO: as duas seções do params.yaml que o treino lê
    - ...
    outs:
    - models

## 4. Rodar: só o que mudou é executado

`prepare` e `split` não mudaram desde a última execução, então o DVC pula os dois e roda só o `train`.

> 💬 Repare na saída do treino: o `train.py` anota o treino no **MLflow** (experimento `bike-demand-pipeline`)
> e cria uma **versão** do modelo `bike-demand`. Essas duas peças são o assunto da T2 e da T3.

In [ ]:
%%bash
dvc dag
# dvc repro

## 5. Guardar o recibo

O `dvc.lock` é o **recibo**: o hash de cada dependência, parâmetro e saída. Ele vai para o Git; as saídas
(dados preparados, treino/teste e modelo) vão para o remote com `dvc push`.

In [ ]:
%%bash
head -20 dvc.lock
# git add dvc.yaml dvc.lock .gitignore data/.gitignore
# git commit -m "Pipeline DVC: prepare -> split -> train"
# dvc push

## 6. Mudar um parâmetro e voltar atrás

Troque `max_leaf_nodes` de 31 para 63 no `params.yaml` (a célula faz isso por você). O DVC percebe que
só o `train` depende desse parâmetro.

In [ ]:
%%bash
# sed -i 's/max_leaf_nodes: 31/max_leaf_nodes: 63/' params.yaml
dvc params diff
# dvc repro

Agora desfaça. O DVC tem um *run-cache*: ele reconhece a combinação antiga e **restaura** o modelo sem treinar de novo.

In [ ]:
%%bash
# git checkout params.yaml
# dvc repro

---
# Parte 2 · Sozinhos

## 1. Escreva os estágios `evaluate` e `plot`

Mesmo molde do `train`, com duas chaves novas: `metrics` e `plots`. Complete `cmd` e `deps` de cada estágio.

| Estágio | Script | Lê | Gera |
|---|---|---|---|
| `evaluate` | `src/evaluate.py` | modelo + teste | `metrics.json` (e anexa as métricas ao run do MLflow) |
| `plot` | `src/plot.py` | modelo + teste | `reports/previsto_vs_real.png` (e anexa a figura ao run) |

In [ ]:
%%writefile dvc.yaml
# Pipeline do projeto. Cada estágio declara o comando (cmd), o que ele lê (deps),
# os parâmetros do params.yaml que usa (params) e o que ele gera (outs).
stages:
  prepare:
    cmd: python src/prepare.py
    deps:
    - src/prepare.py
    - src/bike.py
    - data/raw/bike.csv
    outs:
    - data/prepared

  split:
    cmd: python src/split.py
    deps:
    - src/split.py
    - src/bike.py
    - data/prepared
    params:
    - split
    outs:
    - data/split

  train:
    cmd: python src/train.py
    deps:
    - src/train.py
    - src/bike.py
    - data/split/train.csv
    params:
    - train
    - mlflow
    outs:
    - models

  evaluate:
    # TODO: o comando que roda a avaliação
    cmd: ...
    deps:
    # TODO: o script, o bike.py, o modelo (pasta models) e o arquivo de TESTE
    - ...
    metrics:
    - metrics.json:
        cache: false

  plot:
    # TODO: o comando que gera o gráfico
    cmd: ...
    deps:
    # TODO: o script, o modelo e o arquivo de teste
    - ...
    plots:
    - reports/previsto_vs_real.png:
        cache: false

## 2. Rode o pipeline e veja as métricas (como no passo 4 da Parte 1)

In [ ]:
%%bash
# TODO: veja o grafo, rode o pipeline e mostre as métricas (dvc metrics show)

➡️ Abra `reports/previsto_vs_real.png` no editor: é o real × previsto do modelo no período de teste.

## 3. Guarde o recibo (como no passo 5 da Parte 1)

In [ ]:
%%bash
# TODO: adicione dvc.yaml, dvc.lock, metrics.json e reports/previsto_vs_real.png; faça commit e dvc push

## 4. Um experimento de verdade (como no passo 6 da Parte 1)

No `params.yaml`, mude `max_leaf_nodes` para `15`, ou seja, árvores menores (a célula faz isso por você;
depois, se quiser, teste a sua própria combinação). Rode o pipeline e compare as métricas com o último commit
(`dvc metrics diff`). **Melhorou?** Faça commit. **Piorou?** Desfaça, como no passo 6 da Parte 1.

In [ ]:
%%bash
# sed -i 's/max_leaf_nodes: 31/max_leaf_nodes: 15/' params.yaml
# TODO: mostre a diferença de parâmetros, rode o pipeline e compare as métricas

In [ ]:
%%bash
# TODO: decida: commit (git add params.yaml dvc.lock metrics.json reports/previsto_vs_real.png + git commit + dvc push) ou desfaça (git checkout params.yaml + dvc repro)
# git push